# Structural Probe — Toy Training

Train the probe on a small hardcoded dataset to see how UUAS improves over a random B.

In [ ]:
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import deque
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import minimum_spanning_tree
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW

DEVICE, MODEL_NAME, PROBE_RANK = torch.device('cuda' if torch.cuda.is_available() else 'cpu'), 'gpt2', 64
print(f'Device: {DEVICE}')

---
## Step 1 — Toy dataset

10 sentences with hand-written dependency heads (`0` = ROOT).

In [ ]:
SENTENCES = [
    (['The', 'cat',    'sat',    'on',   'the',  'mat'],       [2,3,0,3,6,4]),
    (['Dogs','chase',  'cats',   'quickly'],                   [2,0,2,2]),
    (['She', 'saw',    'the',    'man',   'with', 'a','telescope'], [2,0,4,2,2,7,5]),
    (['The', 'chef',   'who',    'ran',   'to',  'the','store','is','out'], [2,8,4,2,4,7,5,0,8]),
    (['He',  'quickly','opened', 'the',   'door'],             [3,3,0,5,3]),
    (['The', 'bird',   'sang',   'a',     'sweet','song'],     [2,3,0,6,6,3]),
    (['John','gave',   'Mary',   'a',     'book'],             [2,0,2,5,2]),
    (['The', 'dog',    'that',   'barked','bit',  'the','man'],[2,5,4,2,0,7,5]),
    (['She', 'will',   'read',   'the',   'report'],          [3,3,0,5,3]),
    (['The', 'old',    'man',    'smiled','softly'],           [3,3,4,0,4]),
]
print(f'{len(SENTENCES)} sentences')

---
## Step 2 — Helpers

In [ ]:
def tree_distances(heads):
    N, adj = len(heads), [[] for _ in range(len(heads))]
    for i, h in enumerate(heads):
        if h > 0: adj[i].append(h-1); adj[h-1].append(i)
    dist = np.full((N,N), np.inf); np.fill_diagonal(dist, 0)
    for s in range(N):
        q, seen = deque([(s,0)]), {s}
        while q:
            n, d = q.popleft(); dist[s,n] = d
            for nb in adj[n]:
                if nb not in seen: seen.add(nb); q.append((nb, d+1))
    return dist

def gold_edges(heads):
    return {frozenset([i, h-1]) for i, h in enumerate(heads) if h > 0}

def mst_edges(dist_matrix):
    mst = minimum_spanning_tree(csr_matrix(dist_matrix)).tocoo()
    return {frozenset([int(i), int(j)]) for i, j in zip(mst.row, mst.col)}

def predict_distances(h, B):
    z = h @ B; diff = z.unsqueeze(1) - z.unsqueeze(0)
    return (diff**2).sum(-1)

def uuas(pred, gold):
    return len(pred & gold) / len(gold) if gold else 0.0

def draw_parse(words, heads, edge_set, title='', ax=None, highlight=None):
    if ax is None: _, ax = plt.subplots(figsize=(max(6, len(words)*1.2), 3.2))
    n    = len(words)
    root = next(i for i, h in enumerate(heads) if h == 0)
    for i, w in enumerate(words):
        ax.text(i, 0, w, ha='center', va='center', fontsize=10,
                bbox=dict(boxstyle='round,pad=0.35', fc='#dceeff', ec='#5599cc', lw=1.2))
    ax.text(root, 1.6, 'ROOT', ha='center', va='center', fontsize=9, color='white',
            bbox=dict(boxstyle='round,pad=0.3', fc='#555', ec='#333', lw=1))
    ax.annotate('', xy=(root, 0.28), xytext=(root, 1.42),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))
    for edge in edge_set:
        i, j  = sorted(edge)
        color = 'black' if highlight is None else ('#2ecc71' if edge in highlight else '#e74c3c')
        span  = j - i
        ax.add_patch(mpatches.Arc(((i+j)/2, 0), width=span, height=span*0.55,
                                   angle=0, theta1=0, theta2=180, color=color, lw=1.8))
        ax.annotate('', xy=(j, 0.01), xytext=(j-0.01, span*0.27),
                    arrowprops=dict(arrowstyle='->', color=color, lw=1.5))
    ax.set_xlim(-0.8, n-0.2); ax.set_ylim(-0.5, 2.1)
    ax.axis('off'); ax.set_title(title, fontsize=11)

---
## Step 3 — Extract hidden states (frozen LM)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
lm = AutoModel.from_pretrained(MODEL_NAME, output_hidden_states=True).to(DEVICE).eval()
N_LAYERS, HIDDEN_DIM = lm.config.num_hidden_layers, lm.config.hidden_size
print(f'{MODEL_NAME}: {N_LAYERS} layers, hidden_dim={HIDDEN_DIM}')

@torch.no_grad()
def get_word_states(words):
    enc = tokenizer(words, is_split_into_words=True, return_tensors='pt', add_special_tokens=True).to(DEVICE)
    w2t = {}
    [w2t.setdefault(w,[]).append(t) for t,w in enumerate(enc.word_ids()) if w is not None]
    layers = [torch.stack([hs[0].cpu()[w2t[i]].mean(0) for i in range(len(words))]) for hs in lm(**enc).hidden_states]
    return [layers[0]] + [l - layers[0] for l in layers[1:]]

# Pre-compute and cache
dataset = []
for words, heads in SENTENCES:
    dataset.append(dict(
        words      = words,
        heads      = heads,
        gold_dist  = torch.tensor(tree_distances(heads), dtype=torch.float32),
        gold_edges = gold_edges(heads),
        states     = get_word_states(words),
    ))
print(f'Cached hidden states for {len(dataset)} sentences.')

---
## Step 4 — Structural Probe + training

**Loss (Eq. 1):** $\mathcal{L} = \sum_S \frac{1}{|S|^2} \sum_{i,j} |d_{ij} - \|B(h_i - h_j)\|^2|$

In [ ]:
class StructuralProbe(nn.Module):
    def __init__(self, hidden_dim, rank):
        super().__init__()
        self.B = nn.Parameter(torch.zeros(hidden_dim, rank))
        nn.init.uniform_(self.B, -0.05, 0.05)

    def forward(self, h):
        z    = h @ self.B
        diff = z.unsqueeze(1) - z.unsqueeze(0)
        return (diff**2).sum(-1)

    def loss(self, h, gold_dist):
        pred = self(h)
        n    = h.shape[0]
        return (pred - gold_dist).abs().sum() / n**2


def train(probe, dataset, layer, epochs=200, lr=1e-3):
    opt    = AdamW(probe.parameters(), lr=lr)
    losses = []
    for ep in range(epochs):
        probe.train(); ep_loss = 0
        for s in dataset:
            h    = s['states'][layer].to(DEVICE)
            gold = s['gold_dist'].to(DEVICE)
            loss = probe.loss(h, gold)
            opt.zero_grad(); loss.backward(); opt.step()
            ep_loss += loss.item()
        losses.append(ep_loss / len(dataset))
        if (ep+1) % 50 == 0:
            print(f'  Epoch {ep+1:3d}  loss={losses[-1]:.4f}')
    return losses


LAYER = 7
probe = StructuralProbe(HIDDEN_DIM, PROBE_RANK).to(DEVICE)
print(f'Training on layer {LAYER} ...')
losses = train(probe, dataset, LAYER)

plt.figure(figsize=(6,3))
plt.plot(losses); plt.xlabel('Epoch'); plt.ylabel('L1 loss')
plt.title('Training loss'); plt.tight_layout(); plt.show()

---
## Step 5 — Random B vs Trained B (UUAS per sentence)

In [ ]:
torch.manual_seed(42)
B_random = torch.randn(HIDDEN_DIM, PROBE_RANK) * 0.05
B_trained = probe.B.detach().cpu()

results = []
for s in dataset:
    h = s['states'][LAYER]
    for label, B in [('random', B_random), ('trained', B_trained)]:
        pdist = predict_distances(h, B).numpy()
        score = uuas(mst_edges(pdist), s['gold_edges'])
        results.append({'sent': ' '.join(s['words']), 'B': label, 'uuas': score})

# Print table
print('{:<42} {:>8} {:>8}'.format('Sentence', 'Random', 'Trained'))
print('-'*60)
sents = [' '.join(s['words']) for s in dataset]
for sent in sents:
    r = next(x['uuas'] for x in results if x['sent']==sent and x['B']=='random')
    t = next(x['uuas'] for x in results if x['sent']==sent and x['B']=='trained')
    print(f'{sent:<42} {r:>8.3f} {t:>8.3f}')
avg_r = np.mean([x['uuas'] for x in results if x['B']=='random'])
avg_t = np.mean([x['uuas'] for x in results if x['B']=='trained'])
print('-'*60)
print('{:<42} {:>8.3f} {:>8.3f}'.format('Mean', avg_r, avg_t))

---
## Step 6 — Parse graph: Gold vs Trained probe (pick a sentence)

In [ ]:
s = dataset[0]   # change index to inspect a different sentence

h          = s['states'][LAYER]
pred_edges = mst_edges(predict_distances(h, B_trained).numpy())
score      = uuas(pred_edges, s['gold_edges'])

fig, axes = plt.subplots(1, 2, figsize=(max(12, len(s['words'])*2.4), 3.5))
draw_parse(s['words'], s['heads'], s['gold_edges'],
           title='Gold parse tree', ax=axes[0])
draw_parse(s['words'], s['heads'], pred_edges,
           title=f'Trained probe (layer {LAYER})  UUAS={score:.2f}',
           ax=axes[1], highlight=s['gold_edges'] & pred_edges)
fig.legend(handles=[mpatches.Patch(color='#2ecc71', label='correct'),
                    mpatches.Patch(color='#e74c3c', label='wrong')],
           loc='lower center', ncol=2, frameon=False)
plt.tight_layout(); plt.show()

---
## Step 7 — UUAS per layer (trained probe on each layer separately)

In [ ]:
layer_scores = {'random': [], 'trained': []}

for layer_idx in range(1, N_LAYERS+1):
    # train a fresh probe on this layer
    p = StructuralProbe(HIDDEN_DIM, PROBE_RANK).to(DEVICE)
    train(p, dataset, layer_idx, epochs=200, lr=1e-3)
    B_t = p.B.detach().cpu()

    correct_r = correct_t = total = 0
    for s in dataset:
        h   = s['states'][layer_idx]
        ge  = s['gold_edges']
        correct_r += len(mst_edges(predict_distances(h, B_random).numpy()) & ge)
        correct_t += len(mst_edges(predict_distances(h, B_t).numpy())      & ge)
        total     += len(ge)
    layer_scores['random'].append(correct_r / total)
    layer_scores['trained'].append(correct_t / total)
    print(f'Layer {layer_idx:2d}  random={layer_scores["random"][-1]:.3f}  trained={layer_scores["trained"][-1]:.3f}')

layers = list(range(1, N_LAYERS+1))
plt.figure(figsize=(9, 3.5))
plt.plot(layers, layer_scores['random'],  'o--', color='gray',      label='random B')
plt.plot(layers, layer_scores['trained'], 'o-',  color='steelblue', label='trained B')
plt.xlabel('Layer'); plt.ylabel('UUAS')
plt.title(f'Random vs Trained probe UUAS per layer — {MODEL_NAME}')
plt.xticks(layers); plt.legend(); plt.tight_layout(); plt.show()